# Classer des intentions avec des encodeurs

Des demandes à un assistant vocal (« réveille-moi à neuf heures », « mets du jazz »), chacune avec l'une de 60 intentions : un vrai problème de classification, avec des classes nombreuses, déséquilibrées et parfois proches. Les données : [MASSIVE](https://github.com/alexa/massive), d'Amazon.

1. Les modèles classiques sur TF-IDF : la référence
2. *Embeddings* figés + régression logistique
3. Affiner un encodeur
4. Garder les résultats pour demain (le TP suivant compare avec un LLM)
5. Bonus : SetFit avec 8 exemples par classe, et la calibration

Ce notebook a besoin d'un GPU : *Exécution > Modifier le type d'exécution > GPU T4*.

## Données

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s massive-intents

In [ ]:
import time
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

train = pd.read_parquet("massive-intents/fr-train.parquet")
test = pd.read_parquet("massive-intents/fr-test.parquet")
labels = sorted(train.label.unique())
print(len(train), "exemples d'entraînement,", len(test), "de test,", len(labels), "intentions")
train.label.value_counts().describe()

`record` mesure et garde les résultats de chaque méthode : le F1 macro (chaque intention compte autant), l'*accuracy*, le temps d'entraînement et le temps de prédiction par texte. `confusions` affiche les paires d'intentions les plus souvent confondues.

In [ ]:
results = {}


def record(name: str, predictions, train_seconds: float, predict_seconds: float) -> None:
    results[name] = {
        "predictions": list(predictions),
        "train_s": train_seconds,
        "ms_per_text": 1000 * predict_seconds / len(test),
    }
    print(
        f"{name:30} F1 macro {f1_score(test.label, predictions, average='macro'):.3f}"
        f"  accuracy {accuracy_score(test.label, predictions):.3f}"
        f"  entraînement {train_seconds:5.0f} s  {results[name]['ms_per_text']:.2f} ms/texte"
    )


def confusions(predictions, n: int = 10) -> list[tuple[tuple[str, str], int]]:
    errors = Counter((t, p) for t, p in zip(test.label, predictions) if t != p)
    return errors.most_common(n)

## 1. La référence : les modèles classiques sur TF-IDF

Les modèles du programme officiel, sur une représentation TF-IDF (mots et paires de mots). Lancez la cellule et lisez les résultats : lequel est le meilleur ? Quelles intentions confond-il ?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2)
X_train = vectorizer.fit_transform(train.text)
X_test = vectorizer.transform(test.text)
for name, model in (
    ("TF-IDF + Naive Bayes", MultinomialNB(alpha=0.1)),
    ("TF-IDF + SVM linéaire", LinearSVC()),
    ("TF-IDF + régression logistique", LogisticRegression(max_iter=2000, C=10)),
    ("TF-IDF + forêt aléatoire", RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)),
):
    start = time.time()
    model.fit(X_train, train.label)
    trained = time.time() - start
    start = time.time()
    predictions = model.predict(X_test)
    record(name, predictions, trained, time.time() - start)

# Les arbres (forêts, gradient boosting), rois des données tabulaires, s'en sortent mal
# sur des milliers de colonnes creuses : le gradient boosting (LightGBM) fait pire encore, et
# diverge ici. Sur du texte, la référence classique est un modèle linéaire sur TF-IDF.

In [ ]:
confusions(results["TF-IDF + SVM linéaire"]["predictions"])

## 2. *Embeddings* figés + régression logistique

Encodez les textes d'entraînement et de test avec Qwen3-Embedding-0.6B (`model.encode(textes, prompt=INSTRUCTION, batch_size=64)`), en précédant chaque texte de l'instruction ci-dessous, qui décrit la tâche au modèle. Entraînez une régression logistique sur les vecteurs, et enregistrez le résultat avec `record` (le temps de prédiction comprend l'encodage des textes de test).

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device="cuda").half()
embedder.max_seq_length = 64
INSTRUCTION = "Instruct: Classify the intent of a request to a voice assistant\nQuery: "

In [ ]:
# Votre code ici

### Solution

In [ ]:
start = time.time()
E_train = embedder.encode(train.text.tolist(), prompt=INSTRUCTION, batch_size=64)
classifier = LogisticRegression(max_iter=2000, C=10).fit(E_train, train.label)
trained = time.time() - start
start = time.time()
E_test = embedder.encode(test.text.tolist(), prompt=INSTRUCTION, batch_size=64)
predictions = classifier.predict(E_test)
record("Embeddings + régression", predictions, trained, time.time() - start)

## 3. Affiner un encodeur

Affinez `ENCODER` (un encodeur pré-entraîné, plus une couche de classification) sur les données d'entraînement, avec `transformers` :

- tokenisez les textes (`max_length=64`) et numérotez les intentions
- `AutoModelForSequenceClassification.from_pretrained(ENCODER, num_labels=60)`
- `TrainingArguments(num_train_epochs=3, learning_rate=5e-5, per_device_train_batch_size=32, fp16=True, warmup_steps=0.1, save_strategy="no", report_to="none")`
- un `Trainer` avec un `DataCollatorWithPadding`

L'entraînement prend quelques minutes : lancez-le, et pendant ce temps relisez les résultats des étapes 1 et 2. Prédisez ensuite les intentions du test (`trainer.predict`) et enregistrez le résultat.

In [ ]:
ENCODER = "jhu-clsp/mmBERT-base"  # multilingue, 2025 ; le meilleur des encodeurs testés sur ces données

In [ ]:
# Votre code ici

### Solution

In [ ]:
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

index = {label: i for i, label in enumerate(labels)}
tokenizer = AutoTokenizer.from_pretrained(ENCODER)


def to_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_dict({"text": df.text.tolist(), "label": [index[label] for label in df.label]})
    return ds.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=64), batched=True, remove_columns=["text"])


model = AutoModelForSequenceClassification.from_pretrained(ENCODER, num_labels=len(labels))
args = TrainingArguments(
    "encoder",
    num_train_epochs=3,
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    fp16=True,
    warmup_steps=0.1,
    logging_steps=100,
    save_strategy="no",
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=to_dataset(train), data_collator=DataCollatorWithPadding(tokenizer))
start = time.time()
trainer.train()
trained = time.time() - start

start = time.time()
logits = trainer.predict(to_dataset(test)).predictions
predictions = [labels[i] for i in logits.argmax(axis=1)]
record("Encodeur affiné", predictions, trained, time.time() - start)

## 4. Garder les résultats pour demain

Demain, le TP suivant compare ces méthodes à un LLM. Enregistrez les prédictions de chaque méthode dans `predictions-fr.csv` (une ligne par texte de test : `id`, `label`, puis une colonne par méthode) et les temps dans `timings-fr.csv`, puis téléchargez les deux fichiers (panneau *Fichiers* à gauche, ou `google.colab.files.download`). Une copie de nos résultats est publiée, au cas où.

In [ ]:
# Votre code ici

### Solution

In [ ]:
predictions = pd.DataFrame({"id": test.id, "label": test.label})
for name, result in results.items():
    predictions[name] = result["predictions"]
predictions.to_csv("predictions-fr.csv", index=False)
timings = pd.DataFrame(
    [(name, r["train_s"], r["ms_per_text"]) for name, r in results.items()],
    columns=["method", "train_s", "ms_per_text"],
)
timings.to_csv("timings-fr.csv", index=False)
timings

## 5. Bonus

**SetFit.** Avec seulement 8 exemples par intention, affinez un modèle d'*embeddings* par contraste puis une tête de classification ([SetFit](https://github.com/huggingface/setfit)). Où se place-t-il, avec 20 fois moins d'étiquettes ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
%pip install -q setfit
from datasets import Dataset as HFDataset
from setfit import SetFitModel, Trainer as SetFitTrainer, TrainingArguments as SetFitArguments

few = train.groupby("label").sample(8, random_state=0, replace=True).drop_duplicates()
setfit_model = SetFitModel.from_pretrained("intfloat/multilingual-e5-base", labels=labels)
setfit_trainer = SetFitTrainer(
    model=setfit_model,
    args=SetFitArguments(batch_size=32, num_epochs=1, num_iterations=10, report_to="none"),
    train_dataset=HFDataset.from_dict({"text": ("query: " + few.text).tolist(), "label": few.label.tolist()}),
)
start = time.time()
setfit_trainer.train()
trained = time.time() - start
start = time.time()
predictions = setfit_model.predict(("query: " + test.text).tolist())
record("SetFit (8 par classe)", list(predictions), trained, time.time() - start)

**Calibration.** Pour la régression sur *embeddings* et l'encodeur affiné, tracez le diagramme de fiabilité : la part de prédictions justes en fonction de la probabilité annoncée (`sklearn.calibration.calibration_curve`, sur « la prédiction est-elle juste ? » et la probabilité de la classe prédite). Lequel est le mieux calibré ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
import matplotlib.pyplot as plt
from scipy.special import softmax
from sklearn.calibration import calibration_curve

probabilities = {
    "Embeddings + régression": classifier.predict_proba(E_test),
    "Encodeur affiné": softmax(logits, axis=1),
}
for name, proba in probabilities.items():
    classes = classifier.classes_ if name.startswith("Embeddings") else np.array(labels)
    confidence = proba.max(axis=1)
    correct = classes[proba.argmax(axis=1)] == test.label.to_numpy()
    accuracy, mean_confidence = calibration_curve(correct, confidence, n_bins=10, strategy="quantile")
    plt.plot(mean_confidence, accuracy, "o-", label=name)
plt.plot([0, 1], [0, 1], "--", color="gray", label="Calibration parfaite")
plt.xlabel("Probabilité annoncée")
plt.ylabel("Part de prédictions justes")
plt.legend()
plt.show()